In [1]:
from simsopt.mhd import Boozer,Vmec
from simsopt.geo import Surface,SurfaceRZFourier

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import neo
from neo import NeoContext,neo_surfaces_from_simsopt_boozer
import time
import numpy as np
from pathlib import Path

from ripplepy import initialize_mgrid_field,set_extcur,trace_fieldline,find_axis,set_trace_parameters,plot_fieldline_3d

# Configuration
full_torus = False
nfp = 3


mgrid_candidates = [
    Path("tests/test_file/mgrid_c09r00.nc"),
    Path("test_file/mgrid_c09r00.nc"),
]
mgrid_path = next((p for p in mgrid_candidates if p.exists()), None)
if mgrid_path is None:
    raise FileNotFoundError(
        "Cannot find mgrid file. Tried: " + ", ".join(str(p) for p in mgrid_candidates)
    )
mgrid_filename = str(mgrid_path)
extcur = None

# mgrid_filename = '/home/zkg/CN_H1_scan_fieldlines/H1_design/coils/mgrid_h1_design.nc'
# extcur = [50000, 5000, 1, -80000, -40000]

initialize_mgrid_field(mgrid_filename, nfp, full_torus=full_torus)
set_extcur(extcur)



✓ Loaded mgrid from 'test_file/mgrid_c09r00.nc' with shape (nr=201, nz=201, nphi=36)
✓ No extcur provided; using rawField initialization completed for 10 coil groups.



array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [2]:
vmec_path = "/home/zkg/ripplepy/tests/test_file/wout_ncsx_c09r00_free.nc"
sur_idx = np.linspace(0.1, 1, 10)

RZ_points = []  # 每个元素是 [R, phi, Z]
for s in sur_idx:
    surface = SurfaceRZFourier.from_wout(vmec_path, s)
    RphiZ = surface.cross_section(phi=0)[0]
    RZ = RphiZ[[0, 2]]   # shape: (Npts, 3)
    RZ_points.append(RZ)             # 第一个点: (3,)

RZ_points = np.asarray(RZ_points)       # shape: (len(sur_idx), 3) [R1,Z1], ...]

R0=1.337

vmec = Vmec(str(vmec_path))
boozer = Boozer(vmec)
boozer.mpol = 72
boozer.ntor = 24
# ns_list =  np.array([2, 3, 4, 5, 6, 7, 8, 9, 10])
# boozer.register(ns_list/100)
boozer.register(np.linspace(0.1, 1.0, 10))
boozer.bx.verbose =True
boozer.run()

neoclass = neo.from_simsopt_boozer(boozer)
ctx = NeoContext()
ctx.set_boozer(neoclass)
surfaces = neo_surfaces_from_simsopt_boozer(boozer)
print('Surfaces from simsopt Boozer:', surfaces)
ctx.set_flux_surfaces(surfaces.tolist())
ctx.set_resolution(theta_n=200, phi_n=200)
ctx.set_mode_limits(max_m_mode=0, max_n_mode=0)
ctx.set_transport_options(
    npart=50,
    multra=1,
    acc_req=0.01,
    no_bins=100,
    nstep_per=50,
    nstep_min=500,
    nstep_max=5000,
    calc_nstep_max=0,
)
ctx.set_switches(ref_swi=2, eout_swi=2, calc_cur=0)
ctx.set_output_options(
    write_progress=0,
    write_output_files=0,
    write_integrate=0,
    write_diagnostic=0,
    suppress_file_io=True,
)

ctx.setup_grids()
ctx.run_all()

got_surfaces = ctx.surface_map()
got_epstot = ctx.epstot_profile()

Read ns=99, mpol=11, ntor=6, mnmax=137, mnmax_nyq=537
compute_surfs (0-based indices):  9 19 29 39 48 58 68 78 88 97
Initializing with mboz=72, nboz=24
ntheta = 290, nzeta = 98, # threads = 12
                   |        outboard (theta=0)      |      inboard (theta=pi)      |
thread js_b js zeta| |B|input  |B|Boozer    Error   | |B|input  |B|Boozer    Error |
------------------------------------------------------------------------------------
   0     0   9   0  1.603e+00  1.603e+00  2.098e-10  1.716e+00  1.716e+00  1.362e-09
                pi  1.568e+00  1.568e+00  1.605e-10  1.633e+00  1.633e+00  1.990e-08
   7     7  78   0  1.497e+00  1.497e+00  5.014e-07  1.880e+00  1.880e+00  8.471e-06
                pi  1.474e+00  1.474e+00  1.240e-07  1.702e+00  1.702e+00  4.678e-05
   1     1  19   0  1.580e+00  1.580e+00  1.573e-07  1.745e+00  1.745e+00  1.933e-08
                pi  1.551e+00  1.551e+00  4.711e-09  1.647e+00  1.647e+00  4.951e-07
   2     2  29   0  1.564e+00  1.564e+00  

In [ ]:
from ripplepy import compute_kg_cylindrical, compute_effective_ripple
ripple_results = []
initial_gradpsi = [1,0,0]
for i in RZ_points:
    fieldline_data=trace_fieldline(i, initial_gradpsi,nturn=100, nphi=360)
    ripple = compute_effective_ripple(fieldline_data,R0)
    ripple_results.append(ripple)

✓ Trace parameters set: nturn=100, nphi=360


TypeError: compute_kg_cylindrical() missing 14 required positional arguments: 'Br', 'Bz', 'Bphi', 'B', 'dBr_dr', 'dBr_dz', 'dBr_dphi', 'dBz_dr', 'dBz_dz', 'dBz_dphi', 'dBphi_dr', 'dBphi_dz', 'dBphi_dphi', and 'gradpsi_mag'

In [ ]:
# ref_epstot = epsilon_eff_arr *2 
ref_epstot = [got_epstot [i] * 1 for i in range(len(got_epstot))]
ref_surfaces = RZ_points[:,0]  # 假设 RZ_points 的第一列是表面索引或半径

print('pyneo:', got_epstot)
print('ripplepy:', ref_epstot)

plt.figure()
plt.plot(ref_surfaces, ref_epstot, label="ripplepy")
plt.plot(ref_surfaces, got_epstot, label="pyneo")
plt.xlabel("R")
plt.ylabel("Epsilon")
plt.legend()
plt.show()